In [ ]:
from pathlib import Path
import re, statistics
import os

In [ ]:
# get the current working directory
working_dir = os.getcwd() 
# load wind speed data as wind_a
file_name_anarede = 'ENERGETICO_Rev1.PWF'
file_name_m_format= 'energetico_refined.m'

In [ ]:
src = os.path.join(working_dir, file_name_anarede) ## pwf anarede format
dst = os.path.join(working_dir, file_name_m_format) ## .m format


In [ ]:

with open(src, "r", encoding="latin-1", errors="ignore") as f:
    lines = f.readlines()

def extract_section(name):
    out = []
    start_idx = None
    for i, ln in enumerate(lines):
        if ln.strip() == name:
            start_idx = i + 1
            break
    if start_idx is None:
        return out
    i = start_idx
    while i < len(lines) and not lines[i].strip().startswith('99999'):
        out.append(lines[i].rstrip("\n"))
        i += 1
    return out

dbars = extract_section("DBAR")
dgers = extract_section("DGER")
dlins = extract_section("DLIN")

def parse_dbar_line(ln):
    m = re.match(r"\s*(\d+)\s+(\S+)\s+(\S+)\s+(.*)", ln)
    if not m:
        return None
    bus = int(m.group(1))
    oetgb = m.group(2)
    name = m.group(3)
    tail = m.group(4)
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?", tail)
    return {"bus": bus, "oetgb": oetgb, "name": name, "nums": nums}

all_bus_ids = set()
bus_vm = {}
bus_va = {}
bus_loads = {}
bus_glcode = {}

for ln in dbars:
    if ln.strip().startswith("(") or not ln.strip():
        continue
    p = parse_dbar_line(ln)
    if not p:
        continue
    bus = p['bus']; all_bus_ids.add(bus)
    nums = p['nums']
    gl = nums[0] if len(nums)>=1 else None
    vm=None; va=None; Pl=None; Ql=None
    if len(nums)>=2:
        try:
            vm_candidate = float(nums[1])
            if abs(vm_candidate) > 50:
                vm = vm_candidate/1000.0
            elif vm_candidate > 5:
                vm = vm_candidate/100.0
            else:
                vm = vm_candidate
        except:
            vm=None
    if len(nums)>=3:
        try:
            va = float(nums[2])
        except:
            va=None
    if len(nums)>=10:
        try:
            Pl = float(nums[8])
            Ql = float(nums[9])
        except:
            Pl=None; Ql=None
    if Pl is None and len(nums)>=2:
        try:
            Pl = float(nums[-2])
            Ql = float(nums[-1])
        except:
            Pl=None; Ql=None
    if Pl is not None:
        Pd = Pl
        if abs(Pd) > 1000:
            Pd = Pd/100.0
        Qd = Ql if Ql is not None else 0.0
        if abs(Qd) > 1000:
            Qd = Qd/100.0
        bus_loads[bus] = (Pd, Qd)
    if vm is not None:
        bus_vm[bus] = vm
    if va is not None:
        bus_va[bus] = va
    bus_glcode[bus] = gl

# DGER
gens = []
for ln in dgers:
    if ln.strip().startswith("(") or not ln.strip():
        continue
    m = re.match(r"\s*(\d+)\s+(.*)", ln)
    if not m:
        continue
    bus = int(m.group(1))
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?", m.group(2))
    Pmin = 0.0; Pmax = 0.0
    if len(nums) >= 2:
        Pmin = float(nums[0]); Pmax = float(nums[1])
    elif len(nums) == 1:
        Pmax = float(nums[0])
    Vg = bus_vm.get(bus, 1.0)
    gens.append([bus, 0.0, 0.0, 9999.0, -9999.0, Vg, 100.0, 1, max(Pmax,0.0), max(Pmin,0.0)])

# DLIN
branches = []
rateA_candidates = []
for ln in dlins:
    if ln.strip().startswith("(") or not ln.strip():
        continue
    ints = re.findall(r"\d+", ln)
    if len(ints) < 2:
        continue
    fbus = int(ints[0]); tbus = int(ints[1])
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?", ln)
    tail_nums = nums[2:]
    r_pct = float(tail_nums[0]) if len(tail_nums) >= 1 else 0.0
    x_pct = float(tail_nums[1]) if len(tail_nums) >= 2 else 0.0
    candidate = None
    for tok in tail_nums:
        try:
            val = float(tok)
            if 5 <= abs(val) <= 5000:
                candidate = abs(val); rateA_candidates.append(candidate); break
        except:
            continue
    rateA = candidate if candidate is not None else 0.0
    r = r_pct/100.0; x = x_pct/100.0
    b = 0.0; ratio=0.0; angle=0.0; status=1
    branches.append([fbus, tbus, r, x, b, rateA, 0.0, 0.0, ratio, angle, status, -360, 360])

rep_rate = statistics.median(rateA_candidates) if rateA_candidates else 0.0

all_buses = sorted(set(list(all_bus_ids) + [g[0] for g in gens] + [br[0] for br in branches] + [br[1] for br in branches]))
bus_rows = []
for b in all_buses:
    type_ = 2 if any(g[0]==b for g in gens) else 1
    Pd, Qd = bus_loads.get(b, (0.0, 0.0))
    Gs=0.0; Bs=0.0; area=1; zone=1
    Vm = bus_vm.get(b, 1.0); Va = bus_va.get(b, 0.0)
    baseKV = 0.0; Vmax=1.1; Vmin=0.9
    bus_rows.append([b, type_, Pd, Qd, Gs, Bs, area, Vm, Va, baseKV, zone, Vmax, Vmin])

gens_updated = []
for g in gens:
    Pmax = g[8]
    g[1] = Pmax if Pmax>0 else 0.0
    gens_updated.append(g)

for br in branches:
    if br[5] == 0.0 and rep_rate>0:
        br[5] = rep_rate

def matpower_array(rows):
    return "\n".join(["\t" + " ".join(str(v) for v in row) + ";" for row in rows])

with open(dst, "w", encoding="utf-8") as f:
    f.write("function mpc = case_from_pwf_refined\n")
    f.write("%% Refined conversion from ANAREDE PWF — heuristic mappings for loads and VM/VA\n")
    f.write("mpc.version = '2';\n")
    f.write("mpc.baseMVA = 100;\n\n")

    f.write("%% bus data\n")
    f.write("%  bus_i type Pd Qd Gs Bs area Vm Va baseKV zone Vmax Vmin\n")
    f.write("mpc.bus = [\n")
    f.write(matpower_array(bus_rows))
    f.write("\n];\n\n")

    f.write("%% generator data\n")
    f.write("%  bus Pg Qg Qmax Qmin Vg mBase status Pmax Pmin\n")
    if gens_updated:
        f.write("mpc.gen = [\n")
        f.write(matpower_array(gens_updated))
        f.write("\n];\n\n")
    else:
        f.write("mpc.gen = [];\n\n")

    f.write("%% branch data\n")
    f.write("%  fbus tbus r x b rateA rateB rateC ratio angle status angmin angmax\n")
    if branches:
        f.write("mpc.branch = [\n")
        f.write(matpower_array(branches))
        f.write("\n];\n\n")
    else:
        f.write("mpc.branch = [];\n\n")

    if gens_updated:
        f.write("%% generator cost data\n")
        f.write("mpc.gencost = [\n")
        for _ in gens_updated:
            f.write("\t2 0 0 3 0 20 0;\n")
        f.write("];\n\n")
